# 06 - Content Evaluation with NLI

## Objective

Sentence embeddings are useful for defining $Z_{\mathrm{content}}(W)=E(W),$ but cosine similarity does not explicitly model contradiction. This notebook therefore adds a separate Natural Language Inference (NLI) evaluator.

The goal is **not** to replace the content representation. Instead, we use an NLI model to evaluate whether the semantic content of an anonymized sentence is preserved.

For an original/anonymized pair $(W_{orig}, W_{anon})$, the NLI model estimates:

$$
P(\mathrm{entailment}),\quad P(\mathrm{neutral}),\quad P(\mathrm{contradiction}).
$$

The main model tested here is `cross-encoder/nli-deberta-v3-base`.

## Why a Cross-Encoder?

Unlike a sentence encoder, a Cross-Encoder receives the two sentences together:

$$
f(W_1,W_2) \rightarrow (p_E,p_N,p_C).
$$

This makes it suitable for directly testing whether the anonymized sentence entails, contradicts, or is neutral with respect to the original sentence.

This model is therefore an **evaluation tool**, not the definition of $Z_{\mathrm{content}}$.

In [3]:
import pandas as pd
from sentence_transformers import CrossEncoder

In [4]:
MODEL_NAME = "cross-encoder/nli-deberta-v3-base"

model = CrossEncoder(MODEL_NAME, device="cpu")
print(f"Loaded: {MODEL_NAME}")

Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 202/202 [00:00<00:00, 9970.46it/s]


Loaded: cross-encoder/nli-deberta-v3-base


## Controlled benchmark

The benchmark contains the same semantic categories used in the sentence-embedding experiments. The lexical-overlap category is especially useful for checking whether the evaluator distinguishes shared words from shared meaning.

In [5]:
benchmark = {
    "paraphrase": [
        ("I am studying statistics at EPFL.", "I study statistics at EPFL."),
        ("I went to the supermarket yesterday.", "Yesterday, I bought some groceries."),
        ("The meeting was postponed until Monday.", "The meeting has been moved to Monday."),
    ],
    "contradiction": [
        ("I am studying statistics at EPFL.", "I do not study statistics at EPFL."),
        ("The interview was recorded in Lausanne.", "The interview was not recorded in Lausanne."),
        ("She works in finance.", "She does not work in finance."),
        ("It rained the whole day.", "It did not rain during the day."),
    ],
    "related": [
        ("The weather is very nice today.", "It rained the whole day."),
        ("I am studying statistics at EPFL.", "I am taking a machine learning course at EPFL."),
        ("I bought a new computer yesterday.", "My laptop stopped working this morning."),
    ],
    "unrelated": [
        ("I am studying statistics at EPFL.", "The football match was cancelled because of the rain."),
        ("My favorite programming language is Python.", "The restaurant closes at eleven tonight."),
        ("I bought a new computer yesterday.", "The train arrived twenty minutes late."),
    ],
    "lexical_overlap": [
        ("The bank is closed because it is Sunday.", "The bank is open because it is Sunday."),
        ("I went to the bank to deposit some money.", "I sat on the bank of the river and watched the water."),
        ("The crane lifted the heavy machine.", "The crane flew over the building."),
        ("I saw a bat flying outside my house.", "He bought a new bat for the baseball game."),
        ("The mouse is under the table.", "I bought a new mouse for my computer."),
    ],
}


## Predict NLI probabilities

The model returns three scores. We convert them to probabilities and identify the predicted class.

In [6]:
def evaluate_nli(model, benchmark):
    rows = []

    for category, pairs in benchmark.items():
        scores = model.predict(pairs, apply_softmax=True, show_progress_bar=False)

        for (s1, s2), probs in zip(pairs, scores):
            labels = model.config.label2id
            # Sentence Transformers normally exposes labels through config.
            # We explicitly handle the common DeBERTa NLI ordering below.
            id2label = {v: k for k, v in labels.items()}
            prob_by_label = {id2label[i].lower(): float(probs[i]) for i in range(len(probs))}

            def get_prob(name):
                for key, value in prob_by_label.items():
                    if name in key:
                        return value
                return float("nan")

            p_entailment = get_prob("entail")
            p_neutral = get_prob("neutral")
            p_contradiction = get_prob("contrad")

            probabilities = {
                "entailment": p_entailment,
                "neutral": p_neutral,
                "contradiction": p_contradiction,
            }
            predicted = max(probabilities, key=probabilities.get)

            rows.append({
                "category": category,
                "sentence_1": s1,
                "sentence_2": s2,
                "entailment": p_entailment,
                "neutral": p_neutral,
                "contradiction": p_contradiction,
                "predicted": predicted,
            })

    return pd.DataFrame(rows)

## Expected behavior

For genuine paraphrases we expect high entailment and low contradiction. For explicit negations we expect high contradiction. Both related but non-equivalent sentences and lexically overlapping sentences with different meanings should not automatically be classified as entailment.

In [7]:
results_df = evaluate_nli(model, benchmark)
display(results_df.round(4))

,category,sentence_1,sentence_2,entailment,neutral,contradiction,predicted
0,paraphrase,I am studying statistics at EPFL.,I study statistics at EPFL.,0.9981,0.0018,0.0001,entailment
1,paraphrase,I went to the supermarket yesterday.,"Yesterday, I bought some groceries.",0.0073,0.9926,0.0000,neutral
2,paraphrase,The meeting was postponed until Monday.,The meeting has been moved to Monday.,0.9979,0.0021,0.0001,entailment
3,contradiction,I am studying statistics at EPFL.,I do not study statistics at EPFL.,0.0000,0.0000,0.9999,contradiction
4,contradiction,The interview was recorded in Lausanne.,The interview was not recorded in Lausanne.,0.0000,0.0001,0.9999,contradiction
5,contradiction,She works in finance.,She does not work in finance.,0.0000,0.0000,0.9999,contradiction
6,contradiction,It rained the whole day.,It did not rain during the day.,0.0000,0.0000,0.9999,contradiction
7,related,The weather is very nice today.,It rained the whole day.,0.0000,0.0001,0.9999,contradiction
8,related,I am studying statistics at EPFL.,I am taking a machine learning course at EPFL.,0.0000,0.0002,0.9998,contradiction
9,related,I bought a new computer yesterday.,My laptop stopped working this morning.,0.0001,0.0167,0.9832,contradiction


## Category-level summary

In [8]:
summary = (
    results_df
    .groupby("category")[["entailment", "neutral", "contradiction"]]
    .mean()
    .round(3)
)
display(summary)

,entailment,neutral,contradiction
category,,,
contradiction,0.000,0.000,1.000
lexical_overlap,0.000,0.414,0.585
paraphrase,0.668,0.332,0.000
related,0.000,0.006,0.994
unrelated,0.001,0.001,0.999


## Interpretation of the results

The model behaves as expected for the main semantic distinctions. Explicit contradictions are classified almost entirely as contradiction (1.000), while paraphrases receive a relatively high entailment score (0.668) and, most importantly, no contradiction probability.

Related sentences are almost entirely classified as contradiction (0.994), showing that semantic relatedness alone is not interpreted as entailment. Similarly, unrelated sentences receive almost no entailment probability (0.001).

The lexical overlap category is more ambiguous: despite having overlapping vocabulary, these sentences are not classified as entailment (0.000), with probabilities split between neutral (0.414) and contradiction (0.585). This suggests that the model does not simply rely on shared words to infer semantic equivalence.

Overall, the results show that the NLI model can distinguish paraphrases from contradictions and does not automatically interpret lexical similarity as entailment. 

## Original vs anonymized sentences

The following cell is the interface we will later use with real ASR/transcription data. Replace the examples with an original transcription and its anonymized transcription.

The important point is that the NLI model receives the **two texts together**, unlike the sentence encoder used for $Z_{\mathrm{content}}$.

In [9]:
original_anonymized = [
    (
        "I am studying statistics at EPFL.",
        "I study statistics at EPFL."
    ),
    (
        "I went to the supermarket yesterday.",
        "Yesterday, I bought some groceries."
    ),
    (
        "I am studying statistics at EPFL.",
        "I do not study statistics at EPFL."
    ),
]

scores = model.predict(original_anonymized, apply_softmax=True, show_progress_bar=False)

comparison = pd.DataFrame({
    "original": [p[0] for p in original_anonymized],
    "anonymized": [p[1] for p in original_anonymized],
    "entailment": scores[:, model.config.label2id["entailment"]],
    "neutral": scores[:, model.config.label2id["neutral"]],
    "contradiction": scores[:, model.config.label2id["contradiction"]],
})

display(comparison.round(4))

,original,anonymized,entailment,neutral,contradiction
0,I am studying statistics at EPFL.,I study statistics at EPFL.,0.9981,0.0018,0.0001
1,I went to the supermarket yesterday.,"Yesterday, I bought some groceries.",0.0073,0.9926,0.0000
2,I am studying statistics at EPFL.,I do not study statistics at EPFL.,0.0000,0.0000,0.9999


## Interpretation for the project

The experiments show that the NLI model provides a useful complementary signal for content preservation.

The model shows a relatively strong ability to identify paraphrases as entailment. Paraphrase pairs receive an average entailment probability of 0.668, with no contradiction probability. This indicates that it generally recognizes when two sentences express the same underlying meaning despite differences in wording.

However, the results also show some limitations. Related but non-equivalent sentences are classified mostly as contradiction (0.994), indicating that the NLI model should not be interpreted as a standalone measure of semantic similarity. This supports the use of NLI together with a sentence-embedding similarity measure.

For the project, **Paraphrase-MPNet** will therefore provide the semantic similarity component $Z_{\mathrm{content}}(W)=E_{\mathrm{para}}(W)$, while the NLI model provides an additional pairwise evaluation of the relationship between the original and anonymized transcripts.

For each original/anonymized pair, we can therefore report:

- cosine similarity between the Paraphrase-MPNet embeddings;
- NLI entailment probability;
- NLI contradiction probability.

Together, these signals provide complementary information about whether the anonymization preserves the original content while potentially modifying speaker-identifying information.